# Block Encoding Interface

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter

import torch

from flagquantum import Circuit
from flagquantum.algorithms.primitives.unitary_block_encoding import UnitaryBlockEncoding
from flagquantum.compiler._component_trace.resource_model import (
    ComponentCost,
    GateCost,
    estimate_component_resources,
)
from flagquantum.core.ir import Instruction
from flagquantum.simulation.unitary import get_unitary

## Metadata and signal state

For a system operator $H$, a block encoding is a unitary $U$ with an ancilla
signal state $|G\rangle$ such that
$$
\left\|H-\alpha(\langle G|\otimes I)U(|G\rangle\otimes I)\right\|\le\epsilon.
$$
The scale $\alpha$ bounds the spectral norm of $H$ and $\epsilon$ bounds the
error. The block declares the widths of the system, ancilla and resource
registers. A separate preparation describes the ancilla signal state. Resource
registers carry additional qubits required by the selected construction.

For a linear combination $H=\sum_l w_l U_l$ with nonnegative weights and
$\alpha=\sum_l w_l$, the SELECT oracle applies $U_l$ in the sector with index
$l$. The PREPARE oracle produces
$|G\rangle=\sum_l\sqrt{w_l/\alpha}|l\rangle$. Projection onto that signal state
recovers the normalized weighted sum.

[Hamiltonian Simulation by Qubitization](https://quantum-journal.org/papers/q-2019-07-12-163/)
and [The power of block-encoded matrix powers](https://arxiv.org/abs/1804.01973)
describe the block-encoding convention.

In [ ]:
def phase_child(ports, workspace, classical_bits, adjoint):
    return (Instruction("sdg" if adjoint else "s", ports["q"]),)

encoding = UnitaryBlockEncoding(
    (("q", 1),), phase_child, alpha=2, ancilla_bitsize=1, resource_bitsize=1
)
assert (encoding.system_bitsize, encoding.ancilla_bitsize, encoding.resource_bitsize) == (1, 1, 1)
assert (encoding.alpha, encoding.epsilon) == (2, 0)
assert encoding.signal_state.selection_width == 1
assert encoding.signal_state.emit((0,), (), (), ()) == ()

### Append the encoding and verify its block

The bound child applies S to the system at wire 1. The ancilla at wire 0
and the resource qubit at wire 2 pass through untouched. The signal
preparation is the identity on the clean ancilla. Selecting the ancilla
and resource sector with both bits zero gives S, which encodes $H=2S$.

In [ ]:
binding = dict(system=(1,), ancilla=(0,), resource=(2,))
circuit = Circuit(3, dtype=torch.complex128)
circuit.append(encoding.make_on(**binding))
matrix = get_unitary(circuit)
identity = torch.eye(2, dtype=torch.complex128)
phase = torch.diag(torch.tensor([1, 1j], dtype=torch.complex128))
torch.testing.assert_close(matrix, torch.kron(torch.kron(identity, phase), identity))

selected = torch.tensor([0, 2])
encoded = encoding.alpha * matrix[selected][:, selected]
torch.testing.assert_close(encoded, 2 * phase)

emitted = Counter(gate.name for gate in circuit.to_ir().instructions)
model = estimate_component_resources(
    encoding.resource_model(ComponentCost("Phase", 1, GateCost(clifford=1)))
)
assert emitted == {"s": 1}
assert model.gates.clifford == 1 and model.peak_logical_qubits == 3
print("Emitted primitives", dict(emitted))
print("Modeled Cliffords and peak qubits", model.gates.clifford, model.peak_logical_qubits)

circuit.append(encoding.make_on(**binding, adjoint=True))
torch.testing.assert_close(get_unitary(circuit), torch.eye(8, dtype=torch.complex128))